# data-cleaning_014

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (73).ipynb`

| Field | Value |
|---|---|
| Section | `data_cleaning` |
| Market | `crypto` |
| Trading style | `unknown` |
| Timeframe | `1m` |
| Code cells | 15 |
| Detected functions | fetch_ohlcv_okx_safe, fetch_funding_okx_safe, merge_funding, calculate_position_sizes, calculate_pnl, run_backtest, generate_tradebook_summary, backtest_core, calculate_metrics, optimize_worker, optimize, plot_results |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ccxt

In [ ]:
import ccxt
import pandas as pd
import time

exchange = ccxt.okx({
    'enableRateLimit': True,
    'timeout': 30000,   # 30 seconds
    'options': {
        'defaultType': 'swap'
    }
})


In [ ]:
def fetch_ohlcv_okx_safe(symbol, timeframe='1m', start='2025-01-01'):
    since = exchange.parse8601(start + 'T00:00:00Z')
    all_data = []
    batch_count = 0

    while True:
        try:
            candles = exchange.fetch_ohlcv(
                symbol=symbol,
                timeframe=timeframe,
                since=since,
                limit=100
            )

            if not candles:
                break

            all_data.extend(candles)
            since = candles[-1][0] + 1
            batch_count += 1

            if batch_count % 10 == 0:
                print(f"{symbol}: fetched {len(all_data)} candles")

            if len(candles) < 100:
                break

            time.sleep(exchange.rateLimit / 1000)

        except Exception as e:
            print("Retrying:", e)
            time.sleep(2)

    df = pd.DataFrame(
        all_data,
        columns=['timestamp','open','high','low','close','volume']
    )

    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    return df


In [ ]:
def fetch_funding_okx_safe(symbol):
    while True:
        try:
            data = exchange.fetch_funding_rate_history(symbol)
            break
        except Exception as e:
            print("Funding retry:", e)
            time.sleep(2)

    df = pd.DataFrame(data)
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    df['funding'] = df['fundingRate'].astype(float)

    return df[['timestamp','funding']]


In [ ]:
def merge_funding(ohlcv, funding):
    df = ohlcv.merge(funding, on='timestamp', how='left')
    df['funding'] = df['funding'].ffill().fillna(0.0)
    return df


In [ ]:
btc_df = merge_funding(
    fetch_ohlcv_okx_safe('BTC-USDT-SWAP'),
    fetch_funding_okx_safe('BTC-USDT-SWAP')
)

eth_df = merge_funding(
    fetch_ohlcv_okx_safe('ETH-USDT-SWAP'),
    fetch_funding_okx_safe('ETH-USDT-SWAP')
)

print(len(btc_df), len(eth_df))
btc_df


In [ ]:
btc_df["ticker"] = "BTC"
eth_df["ticker"] = "ETH"

eth_df

In [ ]:
!pip install pykalman

In [ ]:
from pykalman import KalmanFilter
import numpy as np

df = pd.merge(
    eth_df[['timestamp', 'close', 'funding']],
    btc_df[['timestamp', 'close', 'funding']],
    on='timestamp',
    suffixes=('_eth', '_btc')
)

df = df.sort_values('timestamp').reset_index(drop=True)

df['log_eth'] = np.log(df['close_eth'])
df['log_btc'] = np.log(df['close_btc'])

kf = KalmanFilter(
    transition_matrices=[1],
    observation_matrices=[1],
    initial_state_mean=1,
    initial_state_covariance=1,
    observation_covariance=0.001,
    transition_covariance=0.0001
)

state_means, _ = kf.filter(
    df['log_btc'].values - df['log_eth'].values
)

df['beta'] = state_means.flatten()

df['spread'] = df['log_eth'] - df['beta'] * df['log_btc']

lookback = 120

df['spread_mean'] = df['spread'].ewm(span=lookback).mean()
df['spread_std']  = df['spread'].ewm(span=lookback).std()

df['z'] = (df['spread'] - df['spread_mean']) / df['spread_std']

df['funding_diff'] = df['funding_eth'] - df['funding_btc']

df['effective_z'] = df['z'] * np.sign(df['funding_diff'])

df['effective_z'] = df['effective_z'].shift(1)

# Drop the first row which will have NaN after shifting
df = df.dropna()
df

In [ ]:
df.to_csv('/content/drive/MyDrive/btc_&_eth_prep_data.csv', index=False)

In [ ]:
import pandas as pd
import numpy as np

# Strategy Parameters
ENTRY_Z = 3.0
EXIT_Z = 0.3
MAX_Z = 6.0
MAX_HOLD = 108  # hours
NOTIONAL_SIZE = 10000  # USD per trade
TAKER_FEE = 0.0005  # 0.05% taker fee

def calculate_position_sizes(eth_price, btc_price, beta, notional=NOTIONAL_SIZE):
    """
    Calculate position sizes for market-neutral pairs trade
    Long ETH / Short BTC means: long ETH_notional, short (beta * ETH_notional) of BTC
    """
    btc_notional = beta * notional
    eth_notional = notional

    eth_qty = eth_notional / eth_price
    btc_qty = btc_notional / btc_price

    return eth_qty, btc_qty, eth_notional, btc_notional

def calculate_pnl(position, entry_eth, exit_eth, entry_btc, exit_btc,
                  eth_qty, btc_qty, eth_notional, btc_notional):
    """
    Calculate PnL for pairs trade
    position: 1 = long ETH/short BTC, -1 = short ETH/long BTC
    """
    if position == 1:
        # Long ETH, Short BTC
        eth_pnl = (exit_eth - entry_eth) * eth_qty
        btc_pnl = (entry_btc - exit_btc) * btc_qty
    else:
        # Short ETH, Long BTC
        eth_pnl = (entry_eth - exit_eth) * eth_qty
        btc_pnl = (exit_btc - entry_btc) * btc_qty

    total_pnl = eth_pnl + btc_pnl

    # Calculate fees: 4 trades total (enter ETH, enter BTC, exit ETH, exit BTC)
    # Fee applies to notional value at time of trade
    entry_fees = (eth_notional + btc_notional) * TAKER_FEE
    exit_fees = (eth_qty * exit_eth + btc_qty * exit_btc) * TAKER_FEE
    total_fees = entry_fees + exit_fees

    net_pnl = total_pnl - total_fees

    # Return should be based on capital deployed (use ETH notional as base)
    return_pct = net_pnl / eth_notional

    return {
        'eth_pnl': eth_pnl,
        'btc_pnl': btc_pnl,
        'gross_pnl': total_pnl,
        'fees': total_fees,
        'net_pnl': net_pnl,
        'return_pct': return_pct
    }

def run_backtest(df, lookback=24):
    """
    Run pairs trading backtest on dataframe with columns:
    - close_eth, close_btc, z, effective_z, beta
    """
    position = 0
    entry_state = {}
    trades = []

    for i in range(lookback, len(df)):
        row = df.iloc[i]
        timestamp = df.index[i] if hasattr(df.index, 'name') else i

        # === ENTRY LOGIC ===
        if position == 0:
            if row['effective_z'] > ENTRY_Z:
                # Z-score too high: short the spread (short ETH, long BTC)
                position = -1
                eth_qty, btc_qty, eth_not, btc_not = calculate_position_sizes(
                    row['close_eth'], row['close_btc'], row['beta']
                )
                entry_state = {
                    'position': position,
                    'entry_time': timestamp,
                    'entry_idx': i,
                    'entry_z': row['effective_z'],
                    'entry_eth': row['close_eth'],
                    'entry_btc': row['close_btc'],
                    'eth_qty': eth_qty,
                    'btc_qty': btc_qty,
                    'eth_notional': eth_not,
                    'btc_notional': btc_not,
                    'beta': row['beta']
                }

            elif row['effective_z'] < -ENTRY_Z:
                # Z-score too low: long the spread (long ETH, short BTC)
                position = 1
                eth_qty, btc_qty, eth_not, btc_not = calculate_position_sizes(
                    row['close_eth'], row['close_btc'], row['beta']
                )
                entry_state = {
                    'position': position,
                    'entry_time': timestamp,
                    'entry_idx': i,
                    'entry_z': row['effective_z'],
                    'entry_eth': row['close_eth'],
                    'entry_btc': row['close_btc'],
                    'eth_qty': eth_qty,
                    'btc_qty': btc_qty,
                    'eth_notional': eth_not,
                    'btc_notional': btc_not,
                    'beta': row['beta']
                }

        # === EXIT LOGIC ===
        else:
            hold_time = i - entry_state['entry_idx']

            # Exit conditions
            mean_reversion = abs(row['z']) < EXIT_Z
            stop_loss = abs(row['z']) > MAX_Z
            max_hold_reached = hold_time >= MAX_HOLD

            if mean_reversion or stop_loss or max_hold_reached:
                # Calculate PnL
                pnl_data = calculate_pnl(
                    position,
                    entry_state['entry_eth'],
                    row['close_eth'],
                    entry_state['entry_btc'],
                    row['close_btc'],
                    entry_state['eth_qty'],
                    entry_state['btc_qty'],
                    entry_state['eth_notional'],
                    entry_state['btc_notional']
                )

                # Determine exit reason
                if mean_reversion:
                    exit_reason = 'mean_reversion'
                elif stop_loss:
                    exit_reason = 'stop_loss'
                else:
                    exit_reason = 'max_hold'

                # Record trade
                trades.append({
                    'entry_time': entry_state['entry_time'],
                    'exit_time': timestamp,
                    'entry_idx': entry_state['entry_idx'],
                    'exit_idx': i,
                    'hold_hours': hold_time,
                    'position': 'LONG_ETH_SHORT_BTC' if position == 1 else 'SHORT_ETH_LONG_BTC',
                    'entry_z': entry_state['entry_z'],
                    'exit_z': row['z'],
                    'entry_eth_price': entry_state['entry_eth'],
                    'exit_eth_price': row['close_eth'],
                    'entry_btc_price': entry_state['entry_btc'],
                    'exit_btc_price': row['close_btc'],
                    'eth_qty': entry_state['eth_qty'],
                    'btc_qty': entry_state['btc_qty'],
                    'eth_notional': entry_state['eth_notional'],
                    'btc_notional': entry_state['btc_notional'],
                    'beta': entry_state['beta'],
                    'eth_pnl': pnl_data['eth_pnl'],
                    'btc_pnl': pnl_data['btc_pnl'],
                    'gross_pnl': pnl_data['gross_pnl'],
                    'fees': pnl_data['fees'],
                    'net_pnl': pnl_data['net_pnl'],
                    'return_pct': pnl_data['return_pct'] * 100,
                    'exit_reason': exit_reason
                })

                # Reset position
                position = 0
                entry_state = {}

    return pd.DataFrame(trades)

def generate_tradebook_summary(trade_df):
    """Generate comprehensive trading statistics"""
    if len(trade_df) == 0:
        print("No trades executed")
        return

    print("\n" + "="*70)
    print("PAIRS TRADING BACKTEST RESULTS")
    print("="*70)

    # Basic Stats
    print(f"\nTotal Trades: {len(trade_df)}")
    print(f"Winning Trades: {(trade_df['net_pnl'] > 0).sum()}")
    print(f"Losing Trades: {(trade_df['net_pnl'] < 0).sum()}")
    print(f"Win Rate: {(trade_df['net_pnl'] > 0).mean()*100:.2f}%")

    # PnL Stats
    print(f"\n--- PnL Statistics ---")
    print(f"Total Net PnL: ${trade_df['net_pnl'].sum():.2f}")
    print(f"Average PnL per Trade: ${trade_df['net_pnl'].mean():.2f}")
    print(f"Median PnL per Trade: ${trade_df['net_pnl'].median():.2f}")
    print(f"Best Trade: ${trade_df['net_pnl'].max():.2f}")
    print(f"Worst Trade: ${trade_df['net_pnl'].min():.2f}")

    # Return Stats
    print(f"\n--- Return Statistics ---")
    print(f"Average Return: {trade_df['return_pct'].mean():.3f}%")
    print(f"Median Return: {trade_df['return_pct'].median():.3f}%")
    print(f"Std Dev of Returns: {trade_df['return_pct'].std():.3f}%")

    # Risk Metrics
    if trade_df['return_pct'].std() > 0:
        sharpe = trade_df['return_pct'].mean() / trade_df['return_pct'].std() * np.sqrt(365*24)
        print(f"Sharpe Ratio (annualized): {sharpe:.3f}")

    # Fees
    print(f"\n--- Fees ---")
    print(f"Total Fees Paid: ${trade_df['fees'].sum():.2f}")
    print(f"Average Fees per Trade: ${trade_df['fees'].mean():.2f}")

    # Holding Period
    print(f"\n--- Holding Period ---")
    print(f"Average Hold Time: {trade_df['hold_hours'].mean():.1f} hours")
    print(f"Median Hold Time: {trade_df['hold_hours'].median():.1f} hours")
    print(f"Max Hold Time: {trade_df['hold_hours'].max():.0f} hours")

    # Exit Reasons
    print(f"\n--- Exit Reasons ---")
    exit_counts = trade_df['exit_reason'].value_counts()
    for reason, count in exit_counts.items():
        print(f"{reason}: {count} ({count/len(trade_df)*100:.1f}%)")

    print("\n" + "="*70)

# Example usage (assuming you have a dataframe 'df' with required columns):
trade_df = run_backtest(df, lookback=120)
generate_tradebook_summary(trade_df)

# # Or save to CSV
trade_df.to_csv('tradebook.csv', index=False)

In [ ]:
trade_df

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from numba import jit
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp
from itertools import product

# Constants
NOTIONAL_SIZE = 10000
TAKER_FEE = 0.0005

@jit(nopython=True)
def backtest_core(close_eth, close_btc, z, effective_z, beta,
                  entry_z, exit_z, max_z, max_hold, lookback):
    """JIT-compiled backtest engine"""
    n, position, trade_idx = len(close_eth), 0, 0
    entry_idx, entry_eth, entry_btc = 0, 0.0, 0.0
    eth_qty, btc_qty, eth_not, btc_not = 0.0, 0.0, 0.0, 0.0

    trades = np.zeros(n - lookback)
    equity = np.zeros(n - lookback)
    cumulative = 0.0

    for i in range(lookback, n):
        if position == 0:  # Entry
            if abs(effective_z[i]) > entry_z:
                position = 1 if effective_z[i] < 0 else -1
                eth_not, btc_not = NOTIONAL_SIZE, beta[i] * NOTIONAL_SIZE
                eth_qty, btc_qty = eth_not / close_eth[i], btc_not / close_btc[i]
                entry_idx, entry_eth, entry_btc = i, close_eth[i], close_btc[i]

        else:  # Exit check
            exit_cond = (abs(z[i]) < exit_z or abs(z[i]) > max_z or
                        (i - entry_idx) >= max_hold)

            if exit_cond:
                # PnL calculation
                eth_pnl = (close_eth[i] - entry_eth) * eth_qty * position
                btc_pnl = (entry_btc - close_btc[i]) * btc_qty * position
                fees = (eth_not + btc_not + eth_qty * close_eth[i] +
                       btc_qty * close_btc[i]) * TAKER_FEE

                net_pnl = eth_pnl + btc_pnl - fees
                cumulative += net_pnl

                trades[trade_idx] = net_pnl
                equity[trade_idx] = cumulative
                trade_idx += 1
                position = 0

    return trades[:trade_idx], equity[:trade_idx]

def calculate_metrics(trades, equity):
    """Fast vectorized metrics"""
    if len(trades) < 5:
        return None

    wins = trades[trades > 0]
    losses = trades[trades < 0]

    profit_factor = wins.sum() / abs(losses.sum()) if len(losses) > 0 else 0
    max_dd = (equity - np.maximum.accumulate(equity)).min() if len(equity) > 0 else 0

    # Composite score
    score = (trades.mean() +
            (len(wins) / len(trades)) * 1000 +
            profit_factor * 100 -
            max_dd / 100)

    return {
        'avg_pnl': trades.mean(),
        'win_rate': len(wins) / len(trades),
        'profit_factor': profit_factor,
        'max_drawdown': max_dd,
        'total_trades': len(trades),
        'total_pnl': trades.sum(),
        'composite_score': score
    }

def optimize_worker(args):
    """Parallel worker"""
    entry_z, exit_z, max_z, max_hold, data, lookback = args

    if exit_z >= entry_z or max_z <= entry_z:
        return None

    try:
        trades, equity = backtest_core(*data, entry_z, exit_z, max_z, max_hold, lookback)
        metrics = calculate_metrics(trades, equity)

        if metrics:
            return {
                'entry_z': entry_z, 'exit_z': exit_z,
                'max_z': max_z, 'max_hold': max_hold,
                **metrics
            }
    except:
        pass
    return None

def optimize(df, lookback=24):
    """Main optimization with parallel processing"""
    # Parameter grid
    params = list(product(
        np.arange(2.0, 6.0, 0.2),  # entry_z
        np.arange(0.1, 1.0, 0.2),  # exit_z
        np.arange(5.0, 8.0, 0.5),   # max_z
        [70, 80, 90, 100, 110, 120, 130, 140, 150] # max_hold
    ))

    # Extract data once
    data = (df['close_eth'].values, df['close_btc'].values,
            df['z'].values, df['effective_z'].values, df['beta'].values)

    args = [(e, x, m, h, data, lookback) for e, x, m, h in params]

    print(f"Testing {len(params)} combinations on {mp.cpu_count()} cores...")

    results = []
    with ProcessPoolExecutor(max_workers=mp.cpu_count()) as executor:
        futures = [executor.submit(optimize_worker, arg) for arg in args]
        for future in tqdm(as_completed(futures), total=len(futures)):
            if result := future.result():
                results.append(result)

    return pd.DataFrame(results)

def plot_results(df):
    """Generate comprehensive sensitivity plots"""
    fig = plt.figure(figsize=(20, 15))

    params_info = [
        ('entry_z', 'Entry Z-Score'),
        ('exit_z', 'Exit Z-Score'),
        ('max_z', 'Max Z-Score (Stop Loss)'),
        ('max_hold', 'Max Hold Period (hours)')
    ]

    metrics_info = [
        ('avg_pnl', 'Average PnL ($)', 'blue'),
        ('win_rate', 'Win Rate', 'green'),
        ('profit_factor', 'Profit Factor', 'orange'),
        ('composite_score', 'Composite Score', 'red')
    ]

    # Create 4x4 grid: 4 parameters × 4 metrics
    for i, (param, param_label) in enumerate(params_info):
        for j, (metric, metric_label, color) in enumerate(metrics_info):
            ax = plt.subplot(4, 4, i * 4 + j + 1)

            # Group and plot
            grouped = df.groupby(param)[metric].mean()
            ax.plot(grouped.index, grouped.values, marker='o', linewidth=2.5,
                   markersize=7, color=color, alpha=0.8)

            # Formatting
            ax.set_xlabel(param_label, fontsize=10, fontweight='bold')
            ax.set_ylabel(metric_label, fontsize=10, fontweight='bold')
            ax.set_title(f'{param_label} vs {metric_label}', fontsize=11, fontweight='bold')
            ax.grid(True, alpha=0.3, linestyle='--')

            # Format y-axis for win_rate as percentage
            if metric == 'win_rate':
                ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y*100:.1f}%'))
            elif metric == 'avg_pnl':
                ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'${y:.0f}'))

    plt.suptitle('Parameter Sensitivity Analysis: All Metrics',
                 fontsize=18, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig('optimization_results.png', dpi=300, bbox_inches='tight')
    plt.show()

def print_summary(df):
    """Print optimization summary"""
    best = df.loc[df['composite_score'].idxmax()]

    print("\n" + "="*80)
    print("🏆 BEST PARAMETERS")
    print("="*80)
    print(f"Entry Z: {best['entry_z']:.2f} | Exit Z: {best['exit_z']:.2f} | "
          f"Max Z: {best['max_z']:.2f} | Max Hold: {best['max_hold']:.0f}h")
    print(f"\nComposite Score: {best['composite_score']:.2f}")
    print(f"Avg PnL: ${best['avg_pnl']:.2f} | Win Rate: {best['win_rate']*100:.2f}% | "
          f"PF: {best['profit_factor']:.3f}")
    print(f"Max DD: ${best['max_drawdown']:.2f} | Trades: {best['total_trades']:.0f} | "
          f"Total PnL: ${best['total_pnl']:.2f}")

    print("\n" + "="*80)
    print("TOP 5 COMBINATIONS")
    print("="*80)
    top5 = df.nlargest(5, 'composite_score')
    print(top5[['entry_z', 'exit_z', 'max_z', 'max_hold', 'composite_score',
                'avg_pnl', 'win_rate', 'profit_factor', 'total_trades']].to_string(index=False))
    print("="*80)

def run_final_backtest(df, params, lookback=24):
    """Run detailed backtest with best parameters"""
    data = (df['close_eth'].values, df['close_btc'].values,
            df['z'].values, df['effective_z'].values, df['beta'].values)

    trades, equity = backtest_core(
        *data, params['entry_z'], params['exit_z'],
        params['max_z'], int(params['max_hold']), lookback
    )

    return pd.DataFrame({
        'net_pnl': trades,
        'return_pct': (trades / NOTIONAL_SIZE) * 100
    })

# MAIN EXECUTION
if __name__ == "__main__":
    print("Starting optimization...\n")

    results = optimize(df, lookback=120)
    results.to_csv('optimization_results.csv', index=False)

    print(f"\n✅ Tested {len(results)} valid combinations")
    print_summary(results)

    plot_results(results)
    print("\n✅ Plot saved as 'optimization_results.png'")

    # Final backtest
    best = results.loc[results['composite_score'].idxmax()]
    trades = run_final_backtest(df, best, lookback=120)
    trades.to_csv('optimal_tradebook.csv', index=False)
    print(f"✅ Optimal tradebook saved ({len(trades)} trades)")